# Data inspect & CSV→Parquet conversion

This notebook helps you:

- Ensure `pyarrow` is available (installing into the kernel if needed),
- Read the CSV fallback (`data/backtesting/XAU_USD/H1.csv`),
- Convert it to Parquet (`data/backtesting/XAU_USD/H1.parquet`) and verify,
- Optionally demonstrate a guarded downloader call (requires OANDA credentials in environment).

Run cells in order. If you use VS Code, open this notebook and run the cells with the selected Python kernel from your project `.venv`.

**Note:** this notebook will not print secrets; it only checks for environment variables when attempting the downloader example.

In [1]:
# Section 1 — kernel & python info
import sys, platform
print('sys.executable:', sys.executable)
print('python version:', sys.version)
print('platform:', platform.platform())

sys.executable: /home/joe/anaconda3/bin/python
python version: 3.12.2 | packaged by conda-forge | (main, Feb 16 2024, 20:50:58) [GCC 12.3.0]
platform: Linux-6.5.0-45-generic-x86_64-with-glibc2.35


In [2]:
# Section 2 — ensure pyarrow is installed (uses the kernel's Python)
import importlib, subprocess, sys
def ensure_package(pkg):
    try:
        importlib.import_module(pkg)
        print(pkg, 'is already installed')
        return True
    except Exception:
        print(pkg, 'not found; installing...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg])
        return True
# prefer pyarrow (pandas default engine)
ensure_package('pyarrow')
# optional: python-dotenv if you want to load a .env file inside the notebook
# ensure_package('python-dotenv')

pyarrow is already installed


True

In [3]:
# Section 3 — imports and versions
import pandas as pd, numpy as np, os, json
from pathlib import Path
import pyarrow
print('pandas', pd.__version__)
print('numpy', np.__version__)
print('pyarrow', pyarrow.__version__)

pandas 2.1.4
numpy 1.26.4
pyarrow 16.1.0


In [4]:
# Section 4 — reproducibility & environment checks
import multiprocessing, platform
np.random.seed(42)
CPUS = multiprocessing.cpu_count()
print('CPUs available:', CPUS)
# GPU check (simple, cross-platform)
try:
    import torch
    gpu_available = torch.cuda.is_available()
except Exception:
    gpu_available = False
print('GPU available (torch):', gpu_available)

CPUs available: 20
GPU available (torch): True


In [6]:
# Section 5 — read existing CSV fallback (if present)
csv_path = Path('/home/joe/Desktop/Algo_trading/oanda-trading-system/data/backtesting/XAU_USD/H1.csv')
if not csv_path.exists():
    print('CSV not found at', csv_path)
else:
    df = pd.read_csv(csv_path, parse_dates=['datetime'], index_col='datetime')
    df = df.sort_index()
    print('CSV loaded: rows=', len(df))
    display(df.head())
    display(df.tail())
    print('min, max, rows:', df.index.min(), df.index.max(), len(df))

CSV loaded: rows= 3356


,open,high,low,close,volume
datetime,,,,,
2025-01-01 23:00:00,2625.070,2625.675,2621.775,2623.815,3107
2025-01-02 00:00:00,2623.810,2627.960,2622.765,2625.420,5103
2025-01-02 01:00:00,2625.415,2634.560,2623.895,2633.045,22630
2025-01-02 02:00:00,2633.030,2636.590,2632.365,2634.775,14712
2025-01-02 03:00:00,2634.835,2635.475,2632.315,2633.715,10112


,open,high,low,close,volume
datetime,,,,,
2025-07-28 03:00:00,3335.675,3343.420,3334.560,3343.420,7492
2025-07-28 04:00:00,3343.260,3344.175,3339.525,3339.725,5493
2025-07-28 05:00:00,3339.670,3344.200,3338.085,3343.405,7575
2025-07-28 06:00:00,3343.385,3345.520,3335.850,3340.090,13297
2025-07-28 07:00:00,3340.085,3342.110,3333.960,3335.110,13235


min, max, rows: 2025-01-01 23:00:00 2025-07-28 07:00:00 3356


In [7]:
# Section 6 — write to parquet (if df exists)
parquet_path = Path('/home/joe/Desktop/Algo_trading/oanda-trading-system/data/backtesting/XAU_USD/H1.parquet')
if 'df' not in globals():
    print('No df in memory; skip parquet write')
else:
    parquet_path.parent.mkdir(parents=True, exist_ok=True)
    df.to_parquet(parquet_path, engine='pyarrow', compression='snappy')
    print('Wrote parquet to', parquet_path)
    print('parquet exists:', parquet_path.exists())

Wrote parquet to /home/joe/Desktop/Algo_trading/oanda-trading-system/data/backtesting/XAU_USD/H1.parquet
parquet exists: True


In [8]:
# Section 7 — read parquet back and verify
if parquet_path.exists():
    dfp = pd.read_parquet(parquet_path, engine='pyarrow')
    dfp = dfp.sort_index()
    print('Parquet rows:', len(dfp))
    print('min, max:', dfp.index.min(), dfp.index.max())
    display(dfp.head())
    dfp.info()
else:
    print('Parquet file not found; run previous cell to create it')

Parquet rows: 3356
min, max: 2025-01-01 23:00:00 2025-07-28 07:00:00


,open,high,low,close,volume
datetime,,,,,
2025-01-01 23:00:00,2625.070,2625.675,2621.775,2623.815,3107
2025-01-02 00:00:00,2623.810,2627.960,2622.765,2625.420,5103
2025-01-02 01:00:00,2625.415,2634.560,2623.895,2633.045,22630
2025-01-02 02:00:00,2633.030,2636.590,2632.365,2634.775,14712
2025-01-02 03:00:00,2634.835,2635.475,2632.315,2633.715,10112


<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 3356 entries, 2025-01-01 23:00:00 to 2025-07-28 07:00:00
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   open    3356 non-null   float64
 1   high    3356 non-null   float64
 2   low     3356 non-null   float64
 3   close   3356 non-null   float64
 4   volume  3356 non-null   int64  
dtypes: float64(4), int64(1)
memory usage: 157.3 KB


---
### Optional: run the downloader (guarded)
The cell below demonstrates how to call the project's `OandaDownloader`. It will *only* run if `OANDA_API_TOKEN` is set in the environment. Do not run this cell unless you understand that it will contact OANDA and may consume API quota.

In [ ]:
# Section 8 — guarded downloader example
import os
if os.environ.get('OANDA_API_TOKEN') is None:
    print('OANDA_API_TOKEN not set — skipping downloader example. Add it to your environment or .env before running.')
else:
    print('OANDA token found; running a small downloader test (1 hour range)')
    from backtesting.data.downloader import OandaDownloader
    import datetime as dt
    dl = OandaDownloader()
    start = dt.datetime(2025,7,28,8,0,0)
    end = dt.datetime(2025,7,28,10,0,0)
    df_new = dl.download('XAU_USD', 'H1', start=start, end=end)
    print('Downloader returned rows:', len(df_new))
    display(df_new)

---
## Export, tests and troubleshooting
The remainder of this notebook outlines useful cells for an interactive workflow: exporting the notebook, running tests, saving artifacts, and debugging. You can add/enable them as needed. Below are examples you can copy into cells to run.

In [ ]:
# Example: save artifacts (model/results)
from joblib import dump, load
results_path = Path('artifacts')
results_path.mkdir(exist_ok=True)
# dummy save: save the head of df if present
if 'df' in globals():
    (results_path / 'df_head.csv').write_text(df.head().to_csv())
    print('Saved df_head.csv')
else:
    print('No df to save')

In [ ]:
# Example: run pytest from the notebook (requires pytest installed)
# You can create a tests/ directory and add simple tests for helper functions.
print('To run tests: open a terminal and run: pytest -q')

### Troubleshooting tips
- If imports fail, ensure you selected the correct kernel (the project's `.venv`).
- If `pyarrow` install fails, try installing from a terminal: `./.venv/bin/python -m pip install pyarrow` and then restart the kernel.
- Use `print()` liberally and `df.info()` to inspect dtypes.
- To restart kernel programmatically from a cell (VS Code/Jupyter):
  ```python
  import os, sys
  os.execv(sys.executable, [sys.executable] + sys.argv)
  ```